# SETU Week 2 - BenHalluEval 12K + Real Wiki + RQ1 + Baselines

**Thesis CSE-98 | Kazi Tajrian & Raihan | Supervisor: Mr. Ratul Barua**

## Eta Week 2 er notebook - Week 1 er por chalabe
Week 1 e tumi pipeline test korecho (draft -> claims -> triage -> correction). Ekhon:
- BenHalluEval 12K data loader
- Real Bengali Wikipedia FAISS index (demo replace)
- RQ1: semantic entropy vs self-consistency vs verbalized + ECE/AUROC
- Baselines: Raw SLM, CoT, Uniform RAG, CoVe + BenHalluScore Table 1

**Kaggle Settings (right panel e MUST):**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (OFF thakle github clone fail korbe)

**Run order:** Cell 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7
Time: ~1-1.5 hour (RQ1 100 claims + Benchmark 50 samples)


In [ ]:
# ============================================================
# CELL 1: Setup - repo clone (same as Week1, zip fallback)
# ============================================================
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu scikit-learn rank-bm25 datasets rouge-score tqdm 2>/dev/null

import os, glob, subprocess, sys, shutil
REPO = "A-Selective-Triage-and-Correct-Framework-for-Hallucination-Mitigation-in-Bng-SLM-CSE-98"
URL  = f"https://github.com/einadid/{REPO}.git"
ROOT = "/kaggle/working"

def has_code(p):
    return os.path.isfile(os.path.join(p, "src/pipeline/setu_pipeline.py"))

repo_path = None
if os.path.isdir(os.path.join(ROOT, REPO)):
    shutil.rmtree(os.path.join(ROOT, REPO), ignore_errors=True)
r = subprocess.run(["git", "clone", "--depth", "1", URL, os.path.join(ROOT, REPO)], capture_output=True, text=True)
if r.returncode == 0 and has_code(os.path.join(ROOT, REPO)):
    repo_path = os.path.join(ROOT, REPO)
    print("SETUP: git clone OK (public repo)")
else:
    print("git clone fail:", (r.stderr or "").strip()[:300])
    import zipfile
    hits = glob.glob("/kaggle/input/**/src/pipeline/setu_pipeline.py", recursive=True)
    if not hits:
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            print("unzipping:", z)
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(ROOT)
            except Exception as e:
                print("  skip:", e)
        hits = glob.glob(os.path.join(ROOT, "**", "src", "pipeline", "setu_pipeline.py"), recursive=True)
    if hits:
        repo_path = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
        print("SETUP: code pawa geche /kaggle/input theke")

if repo_path is None:
    raise SystemExit("\n*** Repo setup fail ***\n(a) GitHub repo public koro\n(b) Code -> Download ZIP -> Kaggle Add Input -> Upload")

os.chdir(repo_path)
sys.path.insert(0, repo_path)
print("Repo root:", repo_path)
print("src/ :", sorted(os.listdir("src")))
print("Week2 files:", os.path.exists("src/data/benhallu_eval_loader.py"), os.path.exists("src/retrieval/wiki_builder.py"))


In [ ]:
# ============================================================
# CELL 2: Import - Week1 + Week2 modules
# ============================================================
import torch, os, sys, json, datetime, random
from src.config import SETUConfig
from src.models.slm_generator import SLMGenerator
from src.models.nli_model import MultilingualNLI
from src.retrieval.retriever import BengaliRetriever
from src.pipeline.claim_decomposer import BengaliClaimDecomposer
from src.pipeline.uncertainty_scorer import UncertaintyScorer
from src.pipeline.triage_router import TriageRouter
from src.pipeline.data_driven_corrector import DataDrivenCorrector
from src.pipeline.reasoning_corrector import ReasoningCorrector
from src.pipeline.abstention import AbstentionModule
from src.pipeline.reassembly import ReassemblyModule
from src.pipeline.setu_pipeline import SETUPipeline
# Week2 new
from src.data.benhallu_eval_loader import BenHalluEvalLoader
from src.retrieval.wiki_builder import WikiIndexBuilder
from src.evaluation.rq1_calibration import RQ1CalibrationStudy
from src.evaluation.run_benchmark import BenchmarkRunner
from src.evaluation.benhallu_score import BenHalluScore

print("All SETU modules (Week1+Week2) imported OK")
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB")

cfg = SETUConfig()
cfg.uncertainty.n_samples = 3  # speed er jonno 3, paper e 7
print("\nModel:", cfg.model.name)
print("NLI:", cfg.uncertainty.nli_model)
print("Retriever:", cfg.retrieval.embedding_model)


In [ ]:
# ============================================================
# CELL 3: BenHalluEval 12K Loader
# ============================================================
loader = BenHalluEvalLoader(data_dir="data")
# Quick test: 20 per task = ~220 instances (GQA 20*5=100 + CodeMixed 80 + Summ 60 + Reasoning 20)
# Full: 100 per task = ~1100 instances, 500 per task = ~5500, 1000 per task = 12K (paper)
data = loader.load_all(max_samples_per_task=20)  # change to 100 for real experiment
loader.stats(data)

split = loader.get_dual_track_split(data)
for task, tracks in split.items():
    print(f"Task {task}: Track A (gold) {len(tracks['track_a'])}, Track B (hallu) {len(tracks['track_b'])}")

# Sample dekhbo
for inst in data[:3]:
    print("\n---")
    print(f"ID: {inst.id} | Task: {inst.task_type} | Lang: {inst.language}")
    print(f"Q: {inst.query[:150]}")
    print(f"Gold: {inst.gold_answer[:150]}")
    print(f"Hallu: {inst.hallucinated_answer[:150] if inst.hallucinated_answer else 'None (gold-only)'}")
    print(f"Type: {inst.metadata.get('hallucination_type') if inst.metadata else 'N/A'}")


In [ ]:
# ============================================================
# CELL 4: Real Bengali Wikipedia FAISS Index
# ============================================================
# Option A: Demo (fast, 300 passages, no internet heavy) - 1 min
# Option B: HF Wikipedia 500 articles (~5k passages) - 20-30 min
# Option C: HF 5000 articles (~50k passages) - 2-3 hours

builder = WikiIndexBuilder(cfg)

# Check existing index
stats = builder.get_stats()
print("Existing index stats:", stats)

MODE = "demo"  # change to "hf_500" or "hf_5000" for real

if MODE == "demo":
    print("Building DEMO index (300 passages)...")
    builder.build_demo()
elif MODE == "hf_500":
    print("Building HF Wikipedia 500 articles (~5k passages) - 20-30 min...")
    builder.build_from_hf(lang="bn", num_articles=500, max_passages=5000)
elif MODE == "hf_5000":
    print("Building HF Wikipedia 5000 articles (~50k passages) - 2-3 hours...")
    builder.build_from_hf(lang="bn", num_articles=5000, max_passages=50000)

print("\nAfter build stats:", builder.get_stats())

# Now load retriever (will auto-load index)
ret = BengaliRetriever(cfg)
print("\nRetriever loaded, test retrieval:")
for q in ["ঢাকার জনসংখ্যা কত?", "বাংলাদেশের রাজধানী কোথায়?", "Dhaka population koto?"]:
    res = ret.retrieve_with_fallback(q, top_k=3)
    print(f"Q: {q} | fallback: {res.get('fallback_used')} | max_score: {res.get('max_score',0):.3f}")
    for r in res.get('results', [])[:2]:
        print(f"   -> [{r['score']:.3f}] {r['passage'][:90]}")


In [ ]:
# ============================================================
# CELL 5: Real Model + Pipeline (same as Week1) + RQ1
# ============================================================
# Load SLM + NLI (takes 4-6 min first time)
gen = SLMGenerator(cfg.model.name, use_4bit=True)  # OOM hole 0.5B
nli = MultilingualNLI(cfg.uncertainty.nli_model)
dec = BengaliClaimDecomposer(slm_generator=gen)
unc = UncertaintyScorer(slm_generator=gen, nli_model=nli, config=cfg)
triage = TriageRouter(config=cfg, slm_generator=gen, retriever=ret)
ddc = DataDrivenCorrector(retriever=ret, slm_generator=gen, config=cfg)
rc = ReasoningCorrector(slm_generator=gen, nli_model=nli, config=cfg)
abst = AbstentionModule(config=cfg)
reasm = ReassemblyModule(slm_generator=gen, config=cfg)
pipe = SETUPipeline(gen, dec, unc, triage, ddc, rc, abst, reasm, cfg)

print("Pipeline ready!")

# RQ1 Calibration Study
print("\n=== RQ1: Calibration Study ===")
study = RQ1CalibrationStudy(gen, nli, cfg)
# For speed: 50 claims, for real: 200-500
rq1_results = study.run_on_instances(data, max_instances=50)
report = study.generate_report(rq1_results, save_path="/kaggle/working/rq1_calibration.json")
print("\nRQ1 report saved to /kaggle/working/rq1_calibration.json + .md")


In [ ]:
# ============================================================
# CELL 6: Baselines + SETU Benchmark (BenHalluScore Table 1)
# ============================================================
runner = BenchmarkRunner(gen, ret, nli, cfg, setu_pipeline=pipe)
# For speed: 30 samples per method, for real: 100-200
bench_results = runner.run_full_benchmark(data, max_samples_per_method=30, save_path="/kaggle/working/benchmark_results.json")
print("\nBenchmark done! Files saved to /kaggle/working/benchmark_results.json + .md")
print("\nTable 1 (BenHalluScore) is in benchmark_results.md - download and send to supervisor")


In [ ]:
# ============================================================
# CELL 7: Final Reports for Supervisor (Week2)
# ============================================================
import datetime, os, json
L = []
L.append("# SETU - Week 2 Progress Report")
L.append("")
L.append(f"Date: {datetime.date.today().isoformat()}")
L.append(f"Model: {cfg.model.name} (4-bit)")
L.append(f"Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
L.append("")
L.append("## 1. BenHalluEval 12K Loader")
L.append(f"- Total instances: {len(data)}")
from collections import Counter
task_counts = Counter([i.task_type for i in data])
L.append(f"- By task: {dict(task_counts)}")
L.append("")
L.append("## 2. Wikipedia FAISS Index")
L.append(f"- Stats: {builder.get_stats()}")
L.append("")
L.append("## 3. RQ1 Calibration")
try:
    with open("/kaggle/working/rq1_calibration.json") as f:
        rq1_json = json.load(f)
    L.append(f"- Overall AUROC: {rq1_json['overall']}")
except:
    L.append("- RQ1 JSON not found, see cell 5 output")
L.append("")
L.append("## 4. Benchmark Table 1 (BenHalluScore)")
try:
    with open("/kaggle/working/benchmark_results.json") as f:
        bench_json = json.load(f)
    L.append(f"- Overall: {bench_json['overall']}")
except:
    L.append("- Benchmark JSON not found, see cell 6 output")
L.append("")
L.append("## 5. Next (Week 3)")
L.append("- Ablation study: No decomposition, No triage, etc.")
L.append("- Error analysis + over-abstention")
L.append("- Thesis Chapter 3 writing")

report = "\n".join(L)
open("/kaggle/working/SUPERVISOR_REPORT_week2.md", "w", encoding="utf-8").write(report)
print(report)
print("\n>>> Saved: /kaggle/working/SUPERVISOR_REPORT_week2.md")
print(">>> Also download: /kaggle/working/rq1_calibration.md and benchmark_results.md from Output tab")
